In [1]:
# Cell 1: Cài đặt thư viện cần thiết cho PIPNet, TorchLM, OpenCV & KaggleHub trên Google Colab / Kaggle
import os, sys, platform, subprocess

if platform.system() == "Linux":
    print("Đang cấu hình gói hệ thống trên Linux (Ubuntu/Colab)...")
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 >/dev/null 2>&1",
        shell=True
    )

print("Đang cài đặt các thư viện Python (torchlm, kagglehub, opencv-python, tqdm, pyyaml, scipy)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
    "torchlm", "kagglehub", "opencv-python", "tqdm", "pyyaml", "scipy"
])
print("Cài đặt hoàn tất! Môi trường đã sẵn sàng.")


Đang cấu hình gói hệ thống trên Linux (Ubuntu/Colab)...
Đang cài đặt các thư viện Python (torchlm, kagglehub, opencv-python, tqdm, pyyaml, scipy)...
Cài đặt hoàn tất! Môi trường đã sẵn sàng.


In [2]:
# Cell 2: Tải / Định vị Dataset (Tự động chuyển đổi giữa Local và Kaggle/Colab)
import os
from pathlib import Path

# Danh sách ứng viên đường dẫn dữ liệu
candidates = [
    Path(r"E:\LSTM\data_processed"),
    Path(r"D:\Project\DATN\driver-guardian\ai\data_processed"),
    Path("./kaggle/data-processed-v3/data_processed"),
    Path("/kaggle/input/data-processed-v3/data_processed"),
]

detected_dataset_dir = None
for c in candidates:
    if (c / "dataset_merged_split.csv").is_file():
        detected_dataset_dir = c
        print(f"Đã tìm thấy Dataset tại: {detected_dataset_dir}")
        break

if detected_dataset_dir is None:
    print("Không tìm thấy manifest cục bộ. Đang tải từ KaggleHub...")
    try:
        import kagglehub
        path_origin = kagglehub.dataset_download(
            "nyvantran6634/data-processed-v3",
            output_dir="./kaggle/data-processed-v3"
        )
        for cand in [Path(path_origin) / "data_processed", Path(path_origin), Path("./kaggle/data-processed-v3/data_processed")]:
            if (cand / "dataset_merged_split.csv").is_file():
                detected_dataset_dir = cand
                break
        if detected_dataset_dir is None:
            detected_dataset_dir = Path(path_origin)
        print(f"Tải thành công! Đường dẫn dữ liệu: {detected_dataset_dir}")
    except Exception as e:
        print(f"Cảnh báo khi tải qua kagglehub ({e}). Fallback đường dẫn mặc định.")
        detected_dataset_dir = Path(r"E:\LSTM\data_processed")


Không tìm thấy manifest cục bộ. Đang tải từ KaggleHub...
Using Colab cache for faster access to the 'data-processed-v3' dataset.
Tải thành công! Đường dẫn dữ liệu: /kaggle/input/data-processed-v3/data_processed


In [3]:
# Cell 3: Import các thư viện cần thiết & Cấu hình môi trường PyTorch CUDA
from __future__ import annotations
import csv
import json
import logging
import math
import os
import random
import sys
import tempfile
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

# Monkey patch cho scipy >= 1.14 để tương thích với torchlm metrics
import scipy.integrate
if not hasattr(scipy.integrate, "simps"):
    scipy.integrate.simps = scipy.integrate.simpson

# Đảm bảo mã hóa UTF-8 trên console
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Cấu hình môi trường nạp DLL PyTorch CUDA trên Windows
torch_lib_path = os.path.join(os.path.dirname(sys.executable), "Library", "bin")
if os.path.exists(torch_lib_path):
    os.environ["PATH"] = torch_lib_path + os.pathsep + os.environ.get("PATH", "")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

logger = logging.getLogger("DriverGuardian.LiteFat.Extract")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(name)s: %(message)s"))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)


In [7]:
# Cell 4: Hệ thống cấu hình trung tâm (ExtractConfig)
@dataclass
class ExtractConfig:
    """
    Cấu hình chi tiết cho pipeline trích xuất .pt từ video qua LiteFatDataset.
    Cho phép tùy biến linh hoạt tham số lấy mẫu thời gian, tiền xử lý và lưu cache.
    """
    # 1. Cấu hình thư mục dữ liệu & Manifest
    dataset_dir: str = str(detected_dataset_dir if 'detected_dataset_dir' in globals() else r"E:\LSTM\data_processed")
    manifest_filename: str = "dataset_merged_split.csv"

    # 2. Cấu hình lấy mẫu thời gian (Temporal Sampling)
    # time_interval = 0.1: lấy mẫu sau mỗi 0.1 giây (10 FPS)
    # window_sec = None: trích xuất toàn bộ thời lượng video để khi train có thể cắt cửa sổ ngẫu nhiên
    time_interval: float = 0.05
    window_sec: Optional[float] = None

    # 3. Cấu hình hình học khuôn mặt & PIPNet
    letterbox_size: Tuple[int, int] = (640, 480)
    face_size: Tuple[int, int] = (224, 224)
    pipnet_backbone: str = "resnet18"
    conf_threshold: float = 0.5
    nms_threshold: float = 0.3
    bbox_margin: float = 0.15
    box_smooth_alpha: float = 0.7

    # 4. Cấu hình lưu trữ Cache (.pt)
    cache_dir: str = "LiteFat/cache_pt"
    cache_splits: List[str] = field(default_factory=lambda: ["train", "val", "test"])
    cache_save_every: int = 100       # Lưu tạm atomic save sau mỗi N video
    resume_cache: bool = True         # Tự động tiếp tục từ các video đã có trong file .pt
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42

    def __post_init__(self):
        if isinstance(self.letterbox_size, (list, tuple)) and len(self.letterbox_size) == 2:
            self.letterbox_size = (int(self.letterbox_size[0]), int(self.letterbox_size[1]))
        if isinstance(self.face_size, (list, tuple)) and len(self.face_size) == 2:
            self.face_size = (int(self.face_size[0]), int(self.face_size[1]))

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def save(self, path: Union[str, Path]) -> None:
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        with open(p, "w", encoding="utf-8") as f:
            json.dump(self.to_dict(), f, indent=4, ensure_ascii=False)

    @classmethod
    def load(cls, path: Union[str, Path]) -> "ExtractConfig":
        p = Path(path)
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        return cls(**data)


In [8]:
# Cell 5: Toàn bộ Định nghĩa Mô-đun Tiền Xử Lý, PIPNet Extractor & LiteFatDataset Độc Lập
# Đảm bảo chạy 100% độc lập trên Google Colab / Kaggle mà không cần tải thêm file ultis.py hay dataset.py
import csv
import os
import random
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Dataset

# ---------------------------------------------------------------------
# 1. TIỆN ÍCH LETTERBOX & KHỞI TẠO PIPNET
# ---------------------------------------------------------------------
def _parse_size(size: Optional[Union[int, Tuple[int, int], List[int]]]) -> Tuple[int, int]:
    """Chuẩn hóa kích thước về tuple (width, height)."""
    if size is None:
        return (640, 480)
    if isinstance(size, int):
        return (size, size)
    if isinstance(size, (tuple, list)) and len(size) == 2:
        return (int(size[0]), int(size[1]))
    raise ValueError(f"Kích thước không hợp lệ: {size}. Phải là int hoặc tuple (w, h).")


def letterbox(
        image: np.ndarray,
        new_size: Optional[Union[int, Tuple[int, int], List[int]]] = (640, 480),
        color: Tuple[int, int, int] = (114, 114, 114),
) -> np.ndarray:
    """Resize ảnh giữ nguyên tỷ lệ khung hình gốc với đệm viền về kích thước đích (w, h)."""
    target_w, target_h = _parse_size(new_size)
    if target_w <= 0 or target_h <= 0:
        return image

    h, w = image.shape[:2]
    if w == target_w and h == target_h:
        return image

    scale = min(target_w / float(w), target_h / float(h))
    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (nw, nh), interpolation=interp)

    if nw == target_w and nh == target_h:
        return resized

    canvas = np.full((target_h, target_w, 3), color, dtype=image.dtype)
    x, y = (target_w - nw) // 2, (target_h - nh) // 2
    canvas[y:y + nh, x:x + nw] = resized
    return canvas


def init_pipnet_68_gpu(backbone: str = "resnet18"):
    """Khởi tạo pipeline FaceBoxesV2 + PIPNet (68 điểm chuẩn 300W) trên CUDA / CPU."""
    import torchlm
    from torchlm.tools import faceboxesv2
    from torchlm.models import pipnet

    device = "cuda" if torch.cuda.is_available() else "cpu"
    logger.info(f"Đang khởi tạo PIPNet ({backbone} - 68 điểm 300W) trên thiết bị: {device.upper()}")
    torchlm.runtime.bind(faceboxesv2(device=device))
    torchlm.runtime.bind(
        pipnet(
            backbone=backbone,
            pretrained=True,
            num_nb=10,
            num_lms=68,
            net_stride=32,
            input_size=256,
            meanface_type="300w",
            map_location=device
        )
    )
    logger.info("Đã nạp xong trọng số mô hình PIPNet 68!")


def compute_iou(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """Tính Intersection over Union (IoU) giữa 2 hộp giới hạn [x1, y1, x2, y2]."""
    xa = max(box_a[0], box_b[0])
    ya = max(box_a[1], box_b[1])
    xb = min(box_a[2], box_b[2])
    yb = min(box_a[3], box_b[3])
    inter = max(0.0, xb - xa) * max(0.0, yb - ya)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - inter
    if union <= 0.0:
        return 0.0
    return float(inter / union)


# ---------------------------------------------------------------------
# 2. BỘ TRÍCH XUẤT PIPNET TỐI ƯU HÓA GPU (FAST PIPNET EXTRACTOR)
# ---------------------------------------------------------------------
class FastPIPNetExtractor:
    """
    Bộ trích xuất 68 điểm mốc khuôn mặt và tọa độ điểm cắt khung mặt cho LiteFat:
      1. Batched FaceBoxesV2 trên GPU: Tối ưu với PriorBox anchors và torchvision NMS.
      2. Temporal Driver Tracking: Theo dõi và khóa khuôn mặt tài xế ổn định theo IoU.
      3. Batched PIPNet 300W trên GPU: Giải mã NRM vector hóa qua bảng ánh xạ.
      4. Landmark-Guided BBox Refinement: Kết hợp cực trị mốc và bbox, nội suy tuyến tính & EMA.
      5. Xuất tọa độ điểm cắt crop_boxes [T, 4] dạng nguyên [x1, y1, x2, y2] trong letterbox.
    """
    def __init__(
            self,
            backbone: str = "resnet18",
            device: Optional[str] = None,
            letterbox_size: Tuple[int, int] = (640, 480),
            face_size: Tuple[int, int] = (224, 224),
            conf_threshold: float = 0.5,
            nms_threshold: float = 0.3,
            bbox_margin: float = 0.15,
            box_smooth_alpha: float = 0.7,
    ):
        self.device = torch.device(
            device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.letterbox_size = _parse_size(letterbox_size)
        self.face_size = _parse_size(face_size)
        self.conf_threshold = float(conf_threshold)
        self.nms_threshold = float(nms_threshold)
        self.bbox_margin = float(bbox_margin)
        self.box_smooth_alpha = float(box_smooth_alpha)

        from torchlm.runtime._wrappers import RuntimeWrapper
        if getattr(RuntimeWrapper, "face_base", None) is None or getattr(RuntimeWrapper, "landmarks_base", None) is None:
            init_pipnet_68_gpu(backbone=backbone)

        self.face_base = RuntimeWrapper.face_base
        self.face_net = self.face_base.net.to(self.device)
        self.face_net.eval()

        self.lmk_base = RuntimeWrapper.landmarks_base
        self.lmk_net = self.lmk_base.to(self.device)
        self.lmk_net.eval()

        target_w, target_h = self.letterbox_size
        from torchlm.tools._faceboxesv2 import PriorBox
        pbox = PriorBox(self.face_base.cfg, image_size=(target_h, target_w))
        self.cached_priors = pbox.forward().to(self.device)
        self.cached_variances = self.face_base.cfg["variance"]
        self.cached_scale = torch.tensor(
            [target_w, target_h, target_w, target_h], dtype=torch.float32, device=self.device
        )
        self.faceboxes_mean = torch.tensor(
            [104.0, 117.0, 123.0], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)

        self.pipnet_mean = torch.tensor(
            [0.485, 0.456, 0.406], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)
        self.pipnet_std = torch.tensor(
            [0.229, 0.224, 0.225], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)

        rev1 = self.lmk_base.reverse_index1
        rev2 = self.lmk_base.reverse_index2
        num_nb = self.lmk_base.num_nb
        self.rev_indices = torch.tensor(
            [r1 * num_nb + r2 for r1, r2 in zip(rev1, rev2)],
            dtype=torch.long,
            device=self.device,
        )

    @torch.no_grad()
    def _detect_faces_batched(self, frames: List[np.ndarray], max_chunk: int = 32) -> List[np.ndarray]:
        T = len(frames)
        if T == 0:
            return []
        all_dets: List[np.ndarray] = []
        priors_xy = self.cached_priors[None, :, :2]
        priors_wh = self.cached_priors[None, :, 2:]
        v0, v1 = self.cached_variances[0], self.cached_variances[1]

        for s in range(0, T, max_chunk):
            chunk = frames[s:s + max_chunk]
            cur_T = len(chunk)
            frames_np = np.stack(chunk, axis=0)
            tensor = torch.from_numpy(frames_np).permute(0, 3, 1, 2).to(self.device).float()
            tensor = tensor - self.faceboxes_mean

            loc, conf = self.face_net(tensor)
            scores = conf[:, :, 1]

            boxes_xy = priors_xy + loc[:, :, :2] * v0 * priors_wh
            boxes_wh = priors_wh * torch.exp(loc[:, :, 2:] * v1)
            b_x1y1 = boxes_xy - boxes_wh / 2.0
            b_x2y2 = b_x1y1 + boxes_wh
            decoded_boxes = torch.cat([b_x1y1, b_x2y2], dim=-1) * self.cached_scale

            for b in range(cur_T):
                sc = scores[b]
                mask = sc >= self.conf_threshold
                if not mask.any():
                    all_dets.append(np.empty((0, 5), dtype=np.float32))
                    continue
                bx = decoded_boxes[b, mask]
                sc_valid = sc[mask]
                keep = torchvision.ops.nms(bx, sc_valid, iou_threshold=self.nms_threshold)[:50]
                res = torch.cat([bx[keep], sc_valid[keep, None]], dim=-1).cpu().numpy()
                all_dets.append(res)
        return all_dets

    def _select_driver_faces(
            self,
            all_dets: List[np.ndarray],
            target_w: int,
            target_h: int
    ) -> Tuple[List[Optional[np.ndarray]], List[bool]]:
        T = len(all_dets)
        chosen_bboxes: List[Optional[np.ndarray]] = [None] * T
        found: List[bool] = [False] * T
        prev_box: Optional[np.ndarray] = None
        prev_box_area = 1.0

        for t in range(T):
            dets = all_dets[t]
            if len(dets) == 0:
                continue
            best_idx = 0
            if prev_box is None:
                best_score = -1.0
                for i, d in enumerate(dets):
                    area = max(0.0, d[2] - d[0]) * max(0.0, d[3] - d[1])
                    metric = d[4] * (area ** 0.5)
                    if metric > best_score:
                        best_score = metric
                        best_idx = i
            else:
                best_match = -1.0
                for i, d in enumerate(dets):
                    iou = compute_iou(d[:4], prev_box[:4])
                    area = max(0.0, d[2] - d[0]) * max(0.0, d[3] - d[1])
                    metric = (iou * 2.0) + (d[4] * 0.5) + (min(1.0, area / (prev_box_area + 1e-6)) * 0.3)
                    if metric > best_match:
                        best_match = metric
                        best_idx = i

            chosen = dets[best_idx].copy()
            chosen_bboxes[t] = chosen
            found[t] = True
            prev_box = chosen
            prev_box_area = max(1.0, (prev_box[2] - prev_box[0]) * (prev_box[3] - prev_box[1]))
        return chosen_bboxes, found

    @torch.no_grad()
    def _extract_landmarks_batched(
            self,
            frames: List[np.ndarray],
            chosen_bboxes: List[Optional[np.ndarray]],
            found: List[bool],
    ) -> Tuple[np.ndarray, np.ndarray]:
        T = len(frames)
        target_w, target_h = self.letterbox_size
        extend = 0.2
        valid_indices = [t for t in range(T) if found[t] and chosen_bboxes[t] is not None]

        landmarks_norm = np.ones((T, 68, 3), dtype=np.float32)
        landmarks_px = np.zeros((T, 68, 2), dtype=np.float32)
        if not valid_indices:
            return landmarks_norm, landmarks_px

        crop_coords, crops_resized = [], []
        for idx in valid_indices:
            bbox = chosen_bboxes[idx]
            bx1, by1, bx2, by2 = bbox[:4]
            bw = bx2 - bx1 + 1.0
            bh = by2 - by1 + 1.0
            cx1 = max(0, min(target_w - 2, int(bx1 - bw * extend / 2.0)))
            cy1 = max(0, min(target_h - 2, int(by1 + bh * extend / 2.0)))
            cx2 = max(cx1 + 1, min(target_w, int(bx2 + bw * extend / 2.0)))
            cy2 = max(cy1 + 1, min(target_h, int(by2 + bh * extend / 2.0)))
            crop_w = float(cx2 - cx1)
            crop_h = float(cy2 - cy1)
            crop_coords.append((cx1, cy1, crop_w, crop_h))

            face_crop = frames[idx][cy1:cy2, cx1:cx2, ::-1]
            face_resized = cv2.resize(face_crop, (256, 256), interpolation=cv2.INTER_LINEAR)
            crops_resized.append(face_resized)

        max_crop_chunk = 64
        batched_lms_list = []
        norm_factor = 1.0 * self.lmk_base.input_size / self.lmk_base.net_stride
        num_nb = self.lmk_base.num_nb
        max_len = self.lmk_base.max_len

        for s in range(0, len(crops_resized), max_crop_chunk):
            chunk_crops = crops_resized[s:s + max_crop_chunk]
            crops_np = np.stack(chunk_crops, axis=0)
            crops_tensor = torch.from_numpy(crops_np).permute(0, 3, 1, 2).to(self.device).float() / 255.0
            crops_tensor = (crops_tensor - self.pipnet_mean) / self.pipnet_std

            out_cls, out_x, out_y, out_nb_x, out_nb_y = self.lmk_net(crops_tensor)
            cur_B, num_lms, H_map, W_map = out_cls.shape
            out_cls = out_cls.view(cur_B, num_lms, -1)
            max_ids = torch.argmax(out_cls, dim=2)
            max_ids_col = max_ids.unsqueeze(-1)

            out_x_flat = out_x.view(cur_B, num_lms, -1)
            out_y_flat = out_y.view(cur_B, num_lms, -1)
            x_sel = torch.gather(out_x_flat, 2, max_ids_col).squeeze(2)
            y_sel = torch.gather(out_y_flat, 2, max_ids_col).squeeze(2)

            max_ids_nb = max_ids_col.repeat(1, 1, num_nb).view(cur_B, num_lms * num_nb, 1)
            out_nb_x_flat = out_nb_x.view(cur_B, num_lms * num_nb, -1)
            out_nb_y_flat = out_nb_y.view(cur_B, num_lms * num_nb, -1)
            nb_x_sel = torch.gather(out_nb_x_flat, 2, max_ids_nb).view(cur_B, num_lms, num_nb)
            nb_y_sel = torch.gather(out_nb_y_flat, 2, max_ids_nb).view(cur_B, num_lms, num_nb)

            pred_x = ((max_ids % W_map).float() + x_sel) / norm_factor
            pred_y = (torch.floor(max_ids.float() / W_map) + y_sel) / norm_factor
            pred_nb_x = ((max_ids % W_map).float().unsqueeze(-1) + nb_x_sel) / norm_factor
            pred_nb_y = (torch.floor(max_ids.float() / W_map).unsqueeze(-1) + nb_y_sel) / norm_factor
            pred_nb_x_flat = pred_nb_x.view(cur_B, -1)
            pred_nb_y_flat = pred_nb_y.view(cur_B, -1)

            nb_x_rev = pred_nb_x_flat[:, self.rev_indices].view(cur_B, num_lms, max_len)
            nb_y_rev = pred_nb_y_flat[:, self.rev_indices].view(cur_B, num_lms, max_len)

            final_x = torch.mean(torch.cat([pred_x.unsqueeze(-1), nb_x_rev], dim=-1), dim=-1)
            final_y = torch.mean(torch.cat([pred_y.unsqueeze(-1), nb_y_rev], dim=-1), dim=-1)
            chunk_lms = torch.stack([final_x, final_y], dim=-1).cpu().numpy()
            batched_lms_list.append(chunk_lms)

        batched_lms = np.concatenate(batched_lms_list, axis=0) if batched_lms_list else np.empty((0, 68, 2), dtype=np.float32)
        for i, idx in enumerate(valid_indices):
            cx1, cy1, cw, ch = crop_coords[i]
            score = float(chosen_bboxes[idx][4])
            lm_crop = batched_lms[i].copy()
            abs_x = lm_crop[:, 0] * cw + cx1
            abs_y = lm_crop[:, 1] * ch + cy1
            landmarks_px[idx, :, 0] = abs_x
            landmarks_px[idx, :, 1] = abs_y
            landmarks_norm[idx, :, 0] = abs_x / float(target_w)
            landmarks_norm[idx, :, 1] = abs_y / float(target_h)
            landmarks_norm[idx, :, 2] = score

        return landmarks_norm, landmarks_px

    def compute_face_crop_boxes(
            self,
            chosen_bboxes: List[Optional[np.ndarray]],
            landmarks_px: np.ndarray,
            found: List[bool],
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Tính toán tọa độ điểm cắt khuôn mặt vuông dạng nguyên [T, 4] [x1, y1, x2, y2]."""
        T = len(chosen_bboxes)
        target_w, target_h = self.letterbox_size
        raw_boxes = np.zeros((T, 4), dtype=np.float32)

        for t in range(T):
            if found[t] and chosen_bboxes[t] is not None:
                bx1, by1, bx2, by2 = chosen_bboxes[t][:4]
                lms = landmarks_px[t]
                lx1, ly1 = float(np.min(lms[:, 0])), float(np.min(lms[:, 1]))
                lx2, ly2 = float(np.max(lms[:, 0])), float(np.max(lms[:, 1]))
                fx1 = min(bx1, lx1)
                fy1 = min(by1, ly1)
                fx2 = max(bx2, lx2)
                fy2 = max(by2, ly2)
                fw = fx2 - fx1
                fh = fy2 - fy1
                cx1 = fx1 - fw * self.bbox_margin
                cy1 = fy1 - fh * self.bbox_margin
                cx2 = fx2 + fw * self.bbox_margin
                cy2 = fy2 + fh * self.bbox_margin
                center_x = (cx1 + cx2) / 2.0
                center_y = (cy1 + cy2) / 2.0
                side = max(cx2 - cx1, cy2 - cy1)
                raw_boxes[t] = [
                    center_x - side / 2.0, center_y - side / 2.0,
                    center_x + side / 2.0, center_y + side / 2.0
                ]

        valid_indices = [t for t in range(T) if found[t]]
        if not valid_indices:
            side = min(target_w, target_h) * 0.8
            cx, cy = target_w / 2.0, target_h / 2.0
            default_box = np.array([cx - side / 2.0, cy - side / 2.0, cx + side / 2.0, cy + side / 2.0], dtype=np.float32)
            for t in range(T):
                raw_boxes[t] = default_box
        else:
            first_idx = valid_indices[0]
            for t in range(first_idx):
                raw_boxes[t] = raw_boxes[first_idx]
            last_idx = valid_indices[-1]
            for t in range(last_idx + 1, T):
                raw_boxes[t] = raw_boxes[last_idx]
            for i in range(len(valid_indices) - 1):
                idx_a = valid_indices[i]
                idx_b = valid_indices[i + 1]
                if idx_b - idx_a > 1:
                    box_a = raw_boxes[idx_a]
                    box_b = raw_boxes[idx_b]
                    for t in range(idx_a + 1, idx_b):
                        alpha = float(t - idx_a) / float(idx_b - idx_a)
                        raw_boxes[t] = (1.0 - alpha) * box_a + alpha * box_b

        smoothed_boxes = np.zeros_like(raw_boxes)
        smoothed_boxes[0] = raw_boxes[0]
        alpha = self.box_smooth_alpha
        for t in range(1, T):
            smoothed_boxes[t] = alpha * raw_boxes[t] + (1.0 - alpha) * smoothed_boxes[t - 1]

        crop_boxes = np.zeros((T, 4), dtype=np.int32)
        for t in range(T):
            bx1, by1, bx2, by2 = smoothed_boxes[t]
            x1 = max(0, min(target_w - 2, int(round(bx1))))
            y1 = max(0, min(target_h - 2, int(round(by1))))
            x2 = max(x1 + 1, min(target_w, int(round(bx2))))
            y2 = max(y1 + 1, min(target_h, int(round(by2))))
            crop_boxes[t] = [x1, y1, x2, y2]

        return crop_boxes, smoothed_boxes

    def _crop_faces_for_litefat(
            self,
            frames: List[np.ndarray],
            crop_boxes: np.ndarray,
    ) -> torch.Tensor:
        T = len(frames)
        crop_w, crop_h = self.face_size
        face_crops = np.zeros((T, crop_h, crop_w, 3), dtype=np.uint8)

        for t in range(T):
            x1, y1, x2, y2 = crop_boxes[t]
            crop = frames[t][y1:y2, x1:x2, :]  # BGR
            if crop.size == 0 or crop.shape[0] < 2 or crop.shape[1] < 2:
                crop = frames[t]
            resized = cv2.resize(crop, (crop_w, crop_h), interpolation=cv2.INTER_LINEAR)
            face_crops[t] = resized[:, :, ::-1]  # Chuyển sang RGB

        face_tensor = torch.from_numpy(face_crops).permute(0, 3, 1, 2).float() / 255.0
        norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        norm_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        face_tensor = (face_tensor - norm_mean) / norm_std
        return face_tensor

    def extract_from_frames(
            self,
            frames: List[np.ndarray],
            crop_faces: bool = True,
            return_crop_boxes: bool = False,
    ) -> Union[
        Tuple[Optional[torch.Tensor], torch.Tensor, List[Optional[np.ndarray]], List[bool]],
        Tuple[Optional[torch.Tensor], torch.Tensor, List[Optional[np.ndarray]], List[bool], np.ndarray],
    ]:
        target_w, target_h = self.letterbox_size
        uniform_frames = []
        for f in frames:
            if f.shape[1] != target_w or f.shape[0] != target_h:
                uniform_frames.append(letterbox(f, new_size=(target_w, target_h)))
            else:
                uniform_frames.append(f)

        all_dets = self._detect_faces_batched(uniform_frames)
        chosen_bboxes, found = self._select_driver_faces(all_dets, target_w, target_h)
        landmarks_norm, landmarks_px = self._extract_landmarks_batched(uniform_frames, chosen_bboxes, found)
        crop_boxes, _ = self.compute_face_crop_boxes(chosen_bboxes, landmarks_px, found)

        face_frames_tensor = None
        if crop_faces:
            face_frames_tensor = self._crop_faces_for_litefat(uniform_frames, crop_boxes)

        landmarks_tensor = torch.from_numpy(landmarks_norm).float()
        if return_crop_boxes:
            return face_frames_tensor, landmarks_tensor, chosen_bboxes, found, crop_boxes
        return face_frames_tensor, landmarks_tensor, chosen_bboxes, found


# ---------------------------------------------------------------------
# 3. ĐỌC NHANH KHUNG HÌNH TỪ VIDEO (FAST VIDEO SEEKER)
# ---------------------------------------------------------------------
def fast_read_video_frames(
        video_path: Union[str, Path],
        target_indices: np.ndarray,
        letterbox_size: Tuple[int, int] = (640, 480),
) -> Tuple[List[np.ndarray], float, int]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Không thể mở tệp video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    target_w, target_h = letterbox_size

    frames: List[np.ndarray] = []
    curr_frame = int(target_indices[0])
    cap.set(cv2.CAP_PROP_POS_FRAMES, curr_frame)

    try:
        for target_idx in target_indices:
            target_idx = int(target_idx)
            if target_idx < curr_frame and frames:
                frames.append(frames[-1].copy())
                continue
            while curr_frame < target_idx:
                if not cap.grab():
                    break
                curr_frame += 1
            ret, frame = cap.read()
            if not ret or frame is None:
                break
            lb_frame = letterbox(frame, new_size=(target_w, target_h))
            frames.append(lb_frame)
            curr_frame += 1
    finally:
        cap.release()

    if not frames:
        raise IOError(f"Không đọc được khung hình nào từ: {video_path}")
    while len(frames) < len(target_indices):
        frames.append(frames[-1].copy())
    return frames, fps, total_frames


# ---------------------------------------------------------------------
# 4. LỚP LITEFAT DATASET (HỖ TRỢ TRÍCH XUẤT .PT & TÙY BIẾN CACHE)
# ---------------------------------------------------------------------
class LiteFatDataset(Dataset):
    """
    PyTorch Dataset nạp dữ liệu cho mô hình LiteFat:
      - Đọc video theo time_interval và window_sec.
      - Trích xuất 68 điểm mốc bằng FastPIPNetExtractor (Batched GPU).
      - Hỗ trợ extract_crops_only=True để trích xuất nhanh điểm mốc & tọa độ cắt khung mặt.
      - Đóng gói đầy đủ metadata định danh video (key, video_id, path, split, label, target_indices...).
    """
    def __init__(
            self,
            dataset_dir: Union[str, Path] = r"E:\LSTM\data_processed",
            split: str = "train",
            time_interval: float = 0.1,
            window_sec: Optional[float] = None,
            manifest_filename: str = "dataset_merged_split.csv",
            letterbox_size: Tuple[int, int] = (640, 480),
            face_size: Tuple[int, int] = (224, 224),
            conf_threshold: float = 0.5,
            bbox_margin: float = 0.15,
            return_meta: bool = False,
            return_raw_frames: bool = False,
            use_cache: bool = False,
            cache_dir: Optional[Union[str, Path]] = None,
            extractor: Optional[FastPIPNetExtractor] = None,
            device: Optional[str] = None,
            extract_crops_only: bool = False,
    ):
        split = split.lower().strip()
        if split not in ("train", "val", "test"):
            raise ValueError(f"Split không hợp lệ: '{split}'. Chỉ chấp nhận 'train', 'val', hoặc 'test'.")

        self.root = Path(dataset_dir)
        self.split = split
        self.time_interval = float(time_interval) if time_interval is not None else 0.1
        self.window_sec = float(window_sec) if window_sec is not None else None
        self.manifest_filename = manifest_filename
        self.letterbox_size = _parse_size(letterbox_size)
        self.face_size = _parse_size(face_size)
        self.conf_threshold = conf_threshold
        self.bbox_margin = bbox_margin
        self.return_meta = bool(return_meta)
        self.return_raw_frames = bool(return_raw_frames)
        self.use_cache = bool(use_cache)
        self.cache_dir = Path(cache_dir) if cache_dir is not None else None
        self.extract_crops_only = bool(extract_crops_only)

        self.samples = self._load_samples()
        if len(self.samples) == 0:
            logger.warning(f"Không tìm thấy mẫu video nào trong {self.root} cho split='{self.split}'")

        self.cached_data: Optional[Dict[str, Any]] = None
        if self.use_cache and self.cache_dir is not None:
            self._load_cache()

        if extractor is not None:
            self.extractor = extractor
        else:
            self.extractor = FastPIPNetExtractor(
                backbone="resnet18",
                device=device,
                letterbox_size=self.letterbox_size,
                face_size=self.face_size,
                conf_threshold=self.conf_threshold,
                bbox_margin=self.bbox_margin,
            )

    def _load_samples(self) -> List[Dict[str, Any]]:
        manifest_path = self.root / self.manifest_filename
        samples = []
        if manifest_path.is_file():
            with open(manifest_path, mode="r", encoding="utf-8-sig") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if row.get("split", "").strip().lower() == self.split:
                        rel_path = row.get("rel_path") or row.get("processed_path")
                        if not rel_path:
                            continue
                        norm_rel = Path(rel_path.replace("\\", "/")).as_posix()
                        full_path = self.root / norm_rel
                        samples.append({
                            "key": norm_rel,
                            "path": full_path,
                            "label": int(row["label"]),
                            "video_id": row.get("video_id") or Path(rel_path).stem,
                            "fps": float(row.get("fps", 30.0)),
                            "frame_count": int(float(row.get("frame_count", 0))),
                            "duration_sec": float(row.get("duration_sec", 0.0)),
                        })
        else:
            for label, folder in ((0, "0_alert"), (1, "1_drowsy")):
                split_folder = self.root / self.split / folder
                if split_folder.is_dir():
                    for p in sorted(split_folder.glob("*.mp4")):
                        samples.append({
                            "key": p.relative_to(self.root).as_posix(),
                            "path": p,
                            "label": label,
                            "video_id": p.stem,
                            "fps": 30.0,
                            "frame_count": 0,
                            "duration_sec": 0.0,
                        })
        return samples

    def _load_cache(self) -> None:
        cache_path = self.cache_dir / f"{self.split}.pt"
        if not cache_path.is_file():
            logger.warning(f"[Cache] Không tìm thấy tệp cache {cache_path}. Sẽ fallback trích xuất on-the-fly.")
            return
        try:
            payload = torch.load(cache_path, map_location="cpu", weights_only=False)
            if isinstance(payload, dict) and "samples" in payload:
                self.cached_data = payload["samples"]
            elif isinstance(payload, dict):
                self.cached_data = payload
            logger.info(f"[Cache] Đã nạp thành công cache {cache_path.name} với {len(self.cached_data)} mẫu.")
        except Exception as e:
            logger.warning(f"[Cache] Lỗi khi nạp cache {cache_path}: {e}. Bỏ qua cache.")

    def _get_frame_indices(self, total_frames: int, fps: float) -> np.ndarray:
        if total_frames <= 0:
            return np.array([0], dtype=int)
        fps = fps if fps > 0 else 30.0
        duration_sec = total_frames / fps

        if self.time_interval == 0:
            if self.window_sec is None:
                return np.arange(total_frames, dtype=int)
            else:
                win_sec = min(self.window_sec, duration_sec)
                num_consec = int(round(win_sec * fps))
                num_consec = max(1, min(num_consec, total_frames))
                max_start = max(0, total_frames - num_consec)
                start_idx = random.randint(0, max_start) if (self.split == "train" and max_start > 0) else max_start // 2
                return np.arange(start_idx, start_idx + num_consec, dtype=int)

        if self.window_sec is None:
            timestamps = np.arange(0, duration_sec, self.time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([0.0])
            indices = np.rint(timestamps * fps).astype(int)
            return np.clip(indices, 0, total_frames - 1)
        else:
            win = min(self.window_sec, duration_sec)
            max_start = max(0.0, duration_sec - win)
            start_sec = random.uniform(0.0, max_start) if (self.split == "train" and max_start > 0) else max_start / 2.0
            timestamps = start_sec + np.arange(0, win, self.time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([start_sec])
            indices = np.rint(timestamps * fps).astype(int)
            return np.clip(indices, 0, total_frames - 1)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Union[
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor],
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, Any]],
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, Any], List[np.ndarray]],
    ]:
        sample = self.samples[index]
        key = sample["key"]

        # 1. Đường nhanh: Nạp từ RAM Cache nếu có
        if self.cached_data is not None and key in self.cached_data:
            item = self.cached_data[key]
            face_frames = item.get("face_frames", None)
            landmarks = item["landmarks"]
            label = torch.tensor(sample["label"], dtype=torch.long)
            meta = {
                "key": key,
                "video_id": sample["video_id"],
                "path": str(sample["path"]),
                "split": self.split,
                "label": int(sample["label"]),
                "fps": float(sample["fps"]),
                "num_frames": landmarks.shape[0],
                "cached": True,
            }
            if "crop_boxes" in item:
                meta["crop_boxes"] = item["crop_boxes"]
            if "bboxes" in item:
                meta["bboxes"] = item["bboxes"]
            if "found" in item:
                meta["found"] = item["found"]
            if "target_indices" in item:
                meta["target_indices"] = item["target_indices"]
            if "meta" in item and isinstance(item["meta"], dict):
                meta.update(item["meta"])

            if self.return_meta:
                return face_frames, landmarks, label, meta
            return face_frames, landmarks, label

        # 2. Đường chuẩn: Đọc video và trích xuất on-the-fly
        fps = sample["fps"]
        total_frames = sample["frame_count"]
        if fps <= 0 or total_frames <= 0:
            cap = cv2.VideoCapture(str(sample["path"]))
            fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 300)
            cap.release()
            sample["fps"] = fps
            sample["frame_count"] = total_frames

        target_indices = self._get_frame_indices(total_frames, fps)
        raw_frames, actual_fps, _ = fast_read_video_frames(
            sample["path"], target_indices, letterbox_size=self.letterbox_size
        )

        crop_faces = not self.extract_crops_only
        face_frames, landmarks, bboxes, found, crop_boxes = self.extractor.extract_from_frames(
            raw_frames,
            crop_faces=crop_faces,
            return_crop_boxes=True,
        )
        label = torch.tensor(sample["label"], dtype=torch.long)

        T_frames = len(found)
        bboxes_tensor = torch.zeros((T_frames, 5), dtype=torch.float32)
        for t_idx, b in enumerate(bboxes):
            if b is not None and len(b) >= 4:
                b_arr = np.asarray(b, dtype=np.float32)
                bboxes_tensor[t_idx, :min(5, len(b_arr))] = torch.from_numpy(b_arr[:min(5, len(b_arr))])
        found_tensor = torch.tensor(found, dtype=torch.bool)
        crop_boxes_tensor = torch.from_numpy(crop_boxes).int()

        meta = {
            "key": key,
            "video_id": sample["video_id"],
            "path": str(sample["path"]),
            "split": self.split,
            "label": int(sample["label"]),
            "fps": float(actual_fps),
            "total_frames": int(total_frames),
            "duration_sec": float(total_frames / actual_fps) if actual_fps > 0 else 0.0,
            "target_indices": torch.from_numpy(target_indices).long(),
            "num_frames": int(landmarks.shape[0]),
            "found_faces_ratio": float(sum(found)) / float(max(1, len(found))),
            "letterbox_size": self.letterbox_size,
            "crop_boxes": crop_boxes_tensor,
            "bboxes": bboxes_tensor,
            "found": found_tensor,
            "raw_bboxes_list": bboxes,
        }

        if self.return_raw_frames and self.return_meta:
            return face_frames, landmarks, label, meta, raw_frames
        if self.return_meta:
            return face_frames, landmarks, label, meta
        return face_frames, landmarks, label

print("Đã nạp toàn bộ các thành phần LiteFatDataset, FastPIPNetExtractor, letterbox thành công!")


In [9]:
# Cell 6: Lớp quản lý trích xuất cache hướng đối tượng (LiteFatPtCacheBuilder)
class LiteFatPtCacheBuilder:
    """
    Bộ điều phối trích xuất và lưu cache .pt cho LiteFat:
      1. Khởi tạo LiteFatDataset với extract_crops_only=True để tối ưu tốc độ và bộ nhớ RAM.
      2. Trích xuất 68 điểm mốc, tọa độ điểm cắt khuôn mặt crop_boxes [T, 4], bboxes [T, 5], found [T].
      3. Ghi nhớ Metadata định danh chi tiết cho từng video (key, video_id, path, split, label, fps, target_indices...).
      4. Hỗ trợ Auto-Resume (bỏ qua video đã trích xuất), Atomic Save an toàn chống hỏng file, và kiểm tra tính toàn vẹn (verify).
    """
    @staticmethod
    def seed_everything(seed: int = 42) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    @staticmethod
    def configure_logging(log_file: Union[Path, str]) -> None:
        log_file = Path(log_file)
        log_file.parent.mkdir(parents=True, exist_ok=True)
        logger.setLevel(logging.INFO)
        logger.propagate = False
        for handler in logger.handlers[:]:
            handler.close()
            logger.removeHandler(handler)

        formatter = logging.Formatter("[%(asctime)s][%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
        file_handler = logging.FileHandler(log_file, encoding="utf-8")
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

        stream_handler = logging.StreamHandler(sys.stdout)
        stream_handler.setFormatter(formatter)
        logger.addHandler(stream_handler)

    @staticmethod
    def atomic_save(state: Any, path: Union[str, Path]) -> None:
        """Lưu tệp an toàn (Atomic Save qua file tạm) để tránh hỏng file .pt khi bị ngắt đột ngột."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fd, tmp_file = tempfile.mkstemp(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp")
        try:
            with os.fdopen(fd, "wb") as f:
                torch.save(state, f)
                f.flush()
                os.fsync(f.fileno())
            os.replace(tmp_file, path)
        finally:
            Path(tmp_file).unlink(missing_ok=True)

    @staticmethod
    def verify_cache_file(path: Union[str, Path]) -> Dict[str, Any]:
        """Kiểm tra tính toàn vẹn của một file cache .pt (cấu trúc, shape landmarks [T, 68, 3], crop_boxes [T, 4], bboxes [T, 5])."""
        path = Path(path)
        report: Dict[str, Any] = {"path": str(path), "ok": False, "num_samples": 0, "errors": []}
        if not path.is_file():
            report["errors"].append("Không tìm thấy file")
            return report
        try:
            payload = torch.load(path, map_location="cpu", weights_only=False)
        except Exception as exc:
            report["errors"].append(f"Không đọc được file: {exc}")
            return report

        samples = payload.get("samples", {})
        report["num_samples"] = len(samples)
        total_frames = 0
        for key, item in samples.items():
            lm = item.get("landmarks")
            cb = item.get("crop_boxes")
            bx = item.get("bboxes")
            meta = item.get("meta", {})
            if lm is None or lm.ndim != 3 or lm.shape[1] != 68 or lm.shape[2] != 3:
                report["errors"].append(f"{key}: shape landmarks không hợp lệ (kỳ vọng [T, 68, 3])")
            elif cb is None or cb.ndim != 2 or cb.shape[1] != 4:
                report["errors"].append(f"{key}: shape crop_boxes không hợp lệ (kỳ vọng [T, 4])")
            elif bx is None or bx.ndim != 2 or bx.shape[1] != 5:
                report["errors"].append(f"{key}: shape bboxes không hợp lệ (kỳ vọng [T, 5])")
            elif not torch.isfinite(lm).all():
                report["errors"].append(f"{key}: landmarks có giá trị NaN/Inf")
            elif "video_id" not in meta or "path" not in meta:
                report["errors"].append(f"{key}: thiếu metadata video định danh")
            else:
                total_frames += lm.shape[0]

        report["total_frames"] = total_frames
        report["ok"] = len(report["errors"]) == 0
        return report

    def __init__(self, cfg: Optional[ExtractConfig] = None):
        self.cfg = cfg if cfg is not None else ExtractConfig()
        self.cfg.__post_init__()

        self.cache_dir = Path(self.cfg.cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

        self.configure_logging(self.cache_dir / "extract.log")
        self.seed_everything(self.cfg.seed)
        self.cfg.save(self.cache_dir / "config.json")

        # Khởi tạo 1 bộ trích xuất PIPNet dùng chung cho mọi split để tiết kiệm GPU VRAM
        self.extractor = FastPIPNetExtractor(
            backbone=self.cfg.pipnet_backbone,
            device=self.cfg.device,
            letterbox_size=self.cfg.letterbox_size,
            face_size=self.cfg.face_size,
            conf_threshold=self.cfg.conf_threshold,
            nms_threshold=self.cfg.nms_threshold,
            bbox_margin=self.cfg.bbox_margin,
            box_smooth_alpha=self.cfg.box_smooth_alpha,
        )

        logger.info("=" * 70)
        logger.info("KHỞI TẠO PIPELINE TRÍCH XUẤT .PT CHO LITEFAT DATASET")
        logger.info("=" * 70)
        logger.info(f"Dataset root              : {self.cfg.dataset_dir}")
        logger.info(f"Thư mục cache (.pt)       : {self.cache_dir}")
        logger.info(f"Các split                 : {self.cfg.cache_splits}")
        logger.info(f"Time interval             : {self.cfg.time_interval}s")
        logger.info(f"Window sec                : {self.cfg.window_sec} (None = toàn bộ video)")
        logger.info(f"Letterbox size (w, h)     : {self.cfg.letterbox_size}")
        logger.info(f"Face size (crop)          : {self.cfg.face_size}")
        logger.info(f"Thiết bị trích xuất       : {self.cfg.device}")

    def _new_cache(self, split: str) -> Dict[str, Any]:
        return {
            "format_version": 1,
            "split": split,
            "time_interval": float(self.cfg.time_interval),
            "window_sec": self.cfg.window_sec,
            "letterbox_size": tuple(self.cfg.letterbox_size),
            "face_size": tuple(self.cfg.face_size),
            "bbox_margin": float(self.cfg.bbox_margin),
            "box_smooth_alpha": float(self.cfg.box_smooth_alpha),
            "num_samples": 0,
            "samples": {},
        }

    def _load_existing_cache(self, split: str, path: Path) -> Dict[str, Any]:
        """Nạp file .pt đã có để Auto-Resume; khởi tạo mới nếu chưa có hoặc cấu hình không khớp."""
        if not (self.cfg.resume_cache and path.is_file()):
            return self._new_cache(split)
        try:
            payload = torch.load(path, map_location="cpu", weights_only=False)
        except Exception as exc:
            logger.warning(f"Không đọc được {path.name} ({exc}). Trích xuất lại từ đầu.")
            return self._new_cache(split)

        if abs(float(payload.get("time_interval", -1.0)) - float(self.cfg.time_interval)) > 1e-9:
            logger.warning(f"{path.name} có time_interval khác cấu hình hiện tại. Trích xuất lại từ đầu.")
            return self._new_cache(split)
        cached_lbs = payload.get("letterbox_size", ())
        if tuple(cached_lbs) != tuple(self.cfg.letterbox_size):
            logger.warning(
                f"{path.name} tạo với letterbox_size={cached_lbs} khác cấu hình hiện tại {self.cfg.letterbox_size}. Trích xuất lại từ đầu."
            )
            return self._new_cache(split)
        return payload

    def extract_split(self, split: str) -> Dict[str, Any]:
        """Trích xuất toàn bộ video của một split và lưu {cache_dir}/{split}.pt."""
        path = self.cache_dir / f"{split}.pt"
        ds = LiteFatDataset(
            dataset_dir=self.cfg.dataset_dir,
            split=split,
            time_interval=self.cfg.time_interval,
            window_sec=self.cfg.window_sec,
            manifest_filename=self.cfg.manifest_filename,
            letterbox_size=self.cfg.letterbox_size,
            face_size=self.cfg.face_size,
            conf_threshold=self.cfg.conf_threshold,
            bbox_margin=self.cfg.bbox_margin,
            return_meta=True,
            extract_crops_only=True,
            use_cache=False,
            extractor=self.extractor,
            device=self.cfg.device,
        )

        stats: Dict[str, Any] = {
            "split": split, "total": len(ds), "cached_before": 0, "new": 0, "failed": [],
            "frames": 0, "seconds": 0.0, "interrupted": False, "path": str(path),
        }
        if len(ds) == 0:
            logger.warning(f"Không có mẫu nào trong split '{split}'. Bỏ qua.")
            return stats

        cache = self._load_existing_cache(split, path)
        done = cache["samples"]
        todo_indices = [i for i, s in enumerate(ds.samples) if s["key"] not in done]
        stats["cached_before"] = len(ds) - len(todo_indices)

        logger.info("=" * 60)
        logger.info(f"BẮT ĐẦU TRÍCH XUẤT SPLIT [{split.upper()}]: tổng {len(ds)} | đã có {stats['cached_before']} | cần trích xuất {len(todo_indices)}")
        logger.info("=" * 60)

        if todo_indices:
            t0 = time.time()
            since_save = 0
            try:
                for idx in tqdm(todo_indices, desc=f"Extracting [{split}]", unit="video", dynamic_ncols=True):
                    sample_meta = ds.samples[idx]
                    key = sample_meta["key"]
                    try:
                        _, landmarks, label, meta = ds[idx]
                    except Exception as exc:
                        logger.error(f"Lỗi trích xuất video '{sample_meta['path']}': {exc}")
                        stats["failed"].append({"key": key, "error": str(exc)})
                        continue

                    done[key] = {
                        "landmarks": landmarks.cpu(),                         # Tensor [T, 68, 3]
                        "crop_boxes": meta["crop_boxes"].cpu(),               # Tensor [T, 4] (x1, y1, x2, y2)
                        "bboxes": meta["bboxes"].cpu(),                       # Tensor [T, 5] (x1, y1, x2, y2, conf)
                        "found": meta["found"].cpu(),                         # Tensor [T] (bool)
                        "label": int(label.item()),
                        "meta": {
                            "key": key,
                            "video_id": str(meta["video_id"]),
                            "path": str(meta["path"]),
                            "split": split,
                            "label": int(label.item()),
                            "fps": float(meta["fps"]),
                            "total_frames": int(meta["total_frames"]),
                            "duration_sec": float(meta["duration_sec"]),
                            "target_indices": meta["target_indices"].cpu(),   # Tensor [T]
                            "num_frames": int(meta["num_frames"]),
                            "found_faces_ratio": float(meta["found_faces_ratio"]),
                            "letterbox_size": tuple(meta["letterbox_size"]),
                        }
                    }
                    stats["new"] += 1
                    stats["frames"] += int(landmarks.shape[0])
                    since_save += 1

                    if since_save >= self.cfg.cache_save_every:
                        cache["num_samples"] = len(done)
                        self.atomic_save(cache, path)
                        since_save = 0
            except KeyboardInterrupt:
                stats["interrupted"] = True
                logger.warning("Quá trình trích xuất bị ngắt bởi người dùng. Đang lưu các mẫu đã hoàn tất...")
            finally:
                cache["num_samples"] = len(done)
                self.atomic_save(cache, path)
                stats["seconds"] = time.time() - t0
        else:
            logger.info(f"Split '{split}' đã đủ 100% mẫu trong {path.name}. Bỏ qua trích xuất.")

        stats["size_mb"] = path.stat().st_size / (1024 * 1024) if path.is_file() else 0.0
        return stats

    def fit(self) -> Dict[str, Any]:
        """Trích xuất toàn bộ các split được chỉ định trong cfg.cache_splits."""
        summary: Dict[str, Any] = {}
        start = time.time()
        for split in self.cfg.cache_splits:
            stats = self.extract_split(split)
            stats["verify"] = self.verify_cache_file(self.cache_dir / f"{split}.pt")
            summary[split] = stats
            if stats["interrupted"]:
                break

        logger.info("=" * 70)
        logger.info(f"HOÀN TẤT TRÍCH XUẤT CACHE sau {(time.time() - start) / 60:.2f} phút.")
        for split, st in summary.items():
            vf = st["verify"]
            speed = (st["new"] / st["seconds"]) if st["seconds"] > 0 else 0.0
            fps_ = (st["frames"] / st["seconds"]) if st["seconds"] > 0 else 0.0
            logger.info(
                f"  [{split:5s}] tổng={st['total']} | mới={st['new']} | lỗi={len(st['failed'])} | "
                f"{speed:.2f} video/s ({fps_:.0f} frame/s) | {st.get('size_mb', 0.0):.1f} MB | "
                f"verify={'OK' if vf['ok'] else 'LỖI'} ({vf['num_samples']} mẫu)"
            )
            for err in vf["errors"][:5]:
                logger.warning(f"      - {err}")
        return summary


In [10]:
# Cell 7: Bắt đầu chạy trích xuất toàn bộ Dataset sang các tệp .pt
config = ExtractConfig()

builder = LiteFatPtCacheBuilder(config)
cache_summary = builder.fit()


In [11]:
# Cell 8: Kiểm tra chi tiết và hiển thị mẫu dữ liệu từ file .pt vừa tạo
test_cache_file = Path(config.cache_dir) / "test.pt"
if test_cache_file.is_file():
    data = torch.load(test_cache_file, map_location="cpu", weights_only=False)
    print(f"File: {test_cache_file.name} | Định dạng: v{data.get('format_version')} | Số mẫu: {data.get('num_samples')}")
    
    # Lấy mẫu đầu tiên để kiểm tra
    first_key = next(iter(data["samples"]))
    sample = data["samples"][first_key]
    meta = sample["meta"]
    
    print("=" * 60)
    print(f"Khóa mẫu (key)      : {first_key}")
    print(f"Video ID            : {meta['video_id']}")
    print(f"Đường dẫn gốc       : {meta['path']}")
    print(f"Nhãn (Label)        : {sample['label']} ({'Drowsy' if sample['label'] == 1 else 'Alert'})")
    print(f"FPS                 : {meta['fps']:.2f}")
    print(f"Thời lượng video    : {meta['duration_sec']:.2f}s (Tổng {meta['total_frames']} frames)")
    print(f"Số frame trích xuất : {meta['num_frames']}")
    print(f"Tỷ lệ phát hiện mặt : {meta['found_faces_ratio'] * 100:.1f}%")
    print(f"Shape landmarks     : {sample['landmarks'].shape} (Kỳ vọng: [T, 68, 3])")
    print(f"Shape crop_boxes    : {sample['crop_boxes'].shape} (Kỳ vọng: [T, 4])")
    print(f"Shape bboxes        : {sample['bboxes'].shape} (Kỳ vọng: [T, 5])")
    print(f"Shape found flags   : {sample['found'].shape} (Kỳ vọng: [T])")
    print(f"Shape target_indices: {meta['target_indices'].shape} (Kỳ vọng: [T])")
    print("=" * 60)
    print("Tọa độ crop_boxes 3 frame đầu tiên:")
    print(sample['crop_boxes'][:3].numpy())
    print("Tọa độ bboxes 3 frame đầu tiên:")
    print(sample['bboxes'][:3].numpy())
else:
    print(f"Chưa tìm thấy file {test_cache_file}.")


In [12]:
# Cell 9 (Tùy chọn cho Colab/Kaggle): Nén thư mục cache .pt và sao lưu lên Google Drive
!zip -r litefat_cache_pt.zip LiteFat/cache_pt
from google.colab import drive
drive.mount('/content/drive')
!cp litefat_cache_pt.zip /content/drive/MyDrive/

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp /content/drive/MyDrive/litefat_cache_pt.zip /content/drive/MyDrive/